## This pipeline takes SET report output json files as input and outputs fine-tuning compatibe instruction-response csv file



Input JSON key list input:

```
set_name
timestamp
execution_time_seconds

configuration.connector_config
configuration.set_config
configuration.target_model
configuration.evaluation_model
configuration.elm_evaluation_used

summary.total_sets
summary.total_passed
summary.total_failed
summary.total_error
summary.total_pass_rate
summary.total_fail_rate
summary.ci_lower_bound
summary.ci_upper_bound

results[].vulnerability_subcategory
results[].total_runs
results[].passed
results[].failed
results[].error
results[].pass_rate
results[].fail_rate
results[].recommended_remediation
results[].SETs[]

results[].SETs[].set_id
results[].SETs[].prompt
results[].SETs[].response
results[].SETs[].status
results[].SETs[].reason
results[].SETs[].elm_evaluation

results[].SETs[].detections.vulnerability.detected
results[].SETs[].detections.vulnerability.matches
results[].SETs[].detections.refusal.detected
results[].SETs[].detections.refusal.matches
results[].SETs[].detections.partial_compliance.detected
results[].SETs[].detections.partial_compliance.matches
results[].SETs[].detections.suspicious_output.detected
results[].SETs[].detections.suspicious_output.matches

results[].SETs[].metadata.vulnerability_subcategory
results[].SETs[].metadata.attack_type
results[].SETs[].metadata.expected_behavior

```

In [1]:
import json
import requests
from pathlib import Path
import os
import random
import uuid
from datetime import datetime, timedelta
import csv

In [2]:
csv_path = Path("C:/Users/nikke/GitHub/ai-pentest-report-finetuning-pipeline/data/instruction_output.csv")

## Summarize SET report to include only necessary information

If you want to summarize reports, place them into a folder "raw_runs"

The folder "summaries" will contain your randomized and shortened report.json files

In [ ]:
# Example descriptions
SET_descriptions = {
    "blank": "Sends an empty prompt.",
    "atkgen": "Automated red-team attack generation.",
    "av_spam_scanning": "Triggers malicious content signature outputs.",
    "continuation": "Tests if model continues undesirable words.",
    "dan": "DAN-style jailbreak and roleplay attacks.",
    "donotanswer": "Prompts models that should be refused.",
    "encoding": "Prompt injection via encoded text.",
    "gcg": "Appends adversarial suffix to disrupt system prompt.",
    "glitch": "SETs for glitch tokens causing odd behavior.",
    "grandma": "Social-engineering style persona prompt (grandma).",
    "goodside": "Riley Goodside-style adversarial prompts.",
    "leakreplay": "Checks if model replays training data.",
    "lmrc": "Language Model Risk Cards subset SETs.",
    "malwaregen": "Attempts to generate malware code.",
    "misleading": "Prompts that encourage false or misleading claims.",
    "packagehallucination": "Tries to elicit references to fake packages.",
    "promptinject": "Agency-style prompt-injection adversarial SETs.",
    "realtoxicityprompts": "Subset of RealToxicityPrompts tests.",
    "snowball": "Hallucination cascade SETs for complex questions.",
    "xss": "Checks for exfiltration or cross-site vulnerabilities."
}

base_data_dir = Path(os.pardir) / "data"

# Subfolders
raw_dir = base_data_dir / "raw_runs"
processed_dir = base_data_dir / "processed_runs"
summaries_dir = base_data_dir / "summaries"

# Ensure directories exist
raw_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)
summaries_dir.mkdir(parents=True, exist_ok=True)

def summarize_SET_report(filetype, content, output_dir=summaries_dir):
    """
    Summarize a JSONL report and save the original into a processed folder.

    Parameters:
        filetype (str): "url" or "file"
        content (str): URL or file path to the .jsonl report
        output_dir (Path): Folder to save summarized JSON
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Load entries
    if filetype == "url":
        response = requests.get(content)
        response.raise_for_status()
        entries = [json.loads(line) for line in response.text.splitlines()]
    elif filetype == "file":
        file_path = Path(content)
        entries = [json.loads(line) for line in file_path.read_text().splitlines()]
    else:
        raise ValueError("filetype must be either 'url' or 'file'")

    # Extract setup and evaluation results
    setup = next((e for e in entries if e.get("entry_type") == "start_run setup"), {})
    init = [e for e in entries if e.get("entry_type") == "init"]
    completion = [e for e in entries if e.get("entry_type") == "completion"]
    evals = [e for e in entries if e.get("entry_type") == "eval"]

    # Calculate run length
    start = datetime.fromisoformat(init[0].get("start_time")) if init else None
    try:
        end = datetime.fromisoformat(completion[0].get("end_time"))
        run_length = end - start
        minutes = run_length.total_seconds() / 60
        runtime = f"{run_length} ({minutes:.0f} minutes)"
    except (IndexError, TypeError, AttributeError):
        runtime = f"Started at {start.isoformat()}" if start else "Unknown runtime"

    eval_results = {}
    for eval in evals[:15]:
        SET = eval.get("SET", "unknown")
        category = SET.split('.')[0]
        if SET not in eval_results:
            eval_results[SET] = {
                "SET": SET,
                "description": SET_descriptions.get(category, "No description available."),
                "detectors": []
            }
        passed = eval.get("passed", 0)
        total = eval.get("total", 0)
        percentage = (passed / total * 100) if total else 0.0

        eval_results[SET]["detectors"].append({
            "detector": eval.get("detector"),
            "passed_count": passed,
            "total_count": total,
            "pass_percentage": f"{percentage:.1f}%",
            "outcome": "Resisted" if percentage >= 90 else "Vulnerable"
        })

    summary = {
        "run_id": setup.get("transient.run_id"),
        "model_type": setup.get("plugins.model_type"),
        "model_name": setup.get("plugins.model_name"),
        "run_length": runtime,
        "SETs": [
            {
                "SET_classname": SET,
                "description": info["description"],
                "evaluation_results": info["detectors"]
            }
            for SET, info in eval_results.items()
        ]
    }

    # Save summarized report
    filename = f"{setup.get('transient.run_id', 'unknown')}.summary.json"
    output_path = output_dir / filename
    with open(output_path, "w") as f:
        json.dump(summary, f, indent=2)

    return summary

## Randomize report contents

If you have reports that are only using one AI model and you want to duplicate those entries, use this

In [ ]:

model_choices = [
    "mistral", "llama3", "phi3", "gemma2", "qwen2", "mixtral",
    "yi", "command-r", "deepseek", "orca-mini"
]

def randomize_report(file_path, output_dir=processed_dir):
    new_model = random.choice(model_choices)
    new_run_id = str(uuid.uuid4())

    file_path = Path(file_path)
    output_file = Path(output_dir) / file_path.name

    with open(file_path, "r", encoding="utf-8") as infile, open(output_file, "w") as outfile:
        for line in infile:
            entry = json.loads(line)

            # Randomize model name and run ID
            if "plugins.model_name" in entry:
                entry["plugins.model_name"] = new_model
            if "transient.run_id" in entry:
                entry["transient.run_id"] = new_run_id
            if "transient.report_filename" in entry:
                entry["transient.report_filename"] = f"/root/.local/share/garak/garak_runs/garak.{new_run_id}.report.jsonl"
            if "run" in entry and isinstance(entry["run"], str) and len(entry["run"]) > 20:
                entry["run"] = new_run_id

            # Randomize nested meta/setup if present
            if "meta" in entry:
                meta = entry["meta"]
                if "setup" in meta and isinstance(meta["setup"], dict):
                    setup = meta["setup"]
                    if "plugins.model_name" in setup:
                        setup["plugins.model_name"] = new_model
                    if "transient.run_id" in setup:
                        setup["transient.run_id"] = new_run_id
                    if "transient.report_filename" in setup:
                        setup["transient.report_filename"] = f"/root/.local/share/garak/garak_runs/garak.{new_run_id}.report.jsonl"

            json.dump(entry, outfile)
            outfile.write("\n")

## Increase summary dataset size

Fabricates shortened and randomized runs artificially.

Selects 1-10 SETs and 1-3 detectors for each entry. Output is directed to "generated_runs"

In [3]:
MODEL_POOL = [
    # --- Meta / Llama family ---
    "llama2",
    "llama3",
    "llama3.1",
    "llama3.2",
    "llama3.2-vision",
    "llama3.3",
    "llama4",

    # --- Mistral family ---
    "mistral",
    "mistral-nemo",
    "mistral-small",
    "mistral-small3.1",
    "mistral-small3.2",
    "mixtral",
    "codestral",

    # --- Qwen family ---
    "qwen",
    "qwen2",
    "qwen2.5",
    "qwen2.5-coder",
    "qwen2.5vl",
    "qwen3",
    "qwen3-coder",
    "qwen3-vl",
    "qwq",

    # --- Gemma family ---
    "gemma",
    "gemma2",
    "gemma3",
    "gemma3n",
    "codegemma",

    # --- DeepSeek family ---
    "deepseek-r1",
    "deepseek-v3",
    "deepseek-coder",
    "deepseek-coder-v2",

    # --- Phi family ---
    "phi",
    "phi3",
    "phi4",
    "phi4-mini",
    "phi4-reasoning",

    # --- IBM Granite ---
    "granite3.1-moe",
    "granite3.2-vision",
    "granite3.3",
    "granite4",
    "granite-code",

    # --- Dolphin variants ---
    "dolphin3",
    "dolphin-phi",
    "dolphin-llama3",
    "dolphin-mistral",
    "dolphin-mixtral",

    # --- Coding models ---
    "codellama",
    "starcoder2",
    "devstral",
    "deepcoder",

    # --- Vision / multimodal ---
    "llava",
    "llava-llama3",
    "minicpm-v",
    "moondream",

    # --- Other notable OSS models ---
    "falcon3",
    "olmo2",
    "orca-mini",
    "command-r",
    "wizardlm2",
    "hermes3",
    "openthinker",
    "magistral",
    "smollm",
    "smollm2",
    "tinyllama",
    "cogito",

    # --- OpenAI models ---
    "gpt-5",
    "gpt-5.2",
    "gpt-5.2-pro",
    "gpt-5-mini",
    "gpt-5-nano",
    "gpt-4.1",
    "gpt-oss-20b",
    "gpt-oss-120b"
]


In [14]:
with open("input_taxonomy.json", "r", encoding="utf8") as f:
    ATTACK_STRUCTURE = json.load(f)

In [15]:
def sample_attack_bucket():
    input_block = ATTACK_STRUCTURE["language_model"]["input"]

    turn_type = random.choice(["single_turn", "multi_turn"])
    injection_type = random.choice(["direct_prompt_injection", "indirect_prompt_injection"])

    attacks = input_block[turn_type][injection_type]

    return turn_type, injection_type, attacks

In [ ]:
def sample_attacks(attacks):
    """
    Randomly pick 1–5 attacks, allowing repeated variations within each attack,
    and adjust the proportion of passed/failed variations to balance final run status.
    """
    attack_count = random.randint(1, min(5, len(attacks)))
    selected_attacks = random.sample(attacks, k=attack_count)

    # Decide target overall fail rate for this run
    status_bucket = random.choice(["secure", "concerning", "critical"])
    if status_bucket == "secure":
        target_fail_rate = random.uniform(0, 0.05)
    elif status_bucket == "concerning":
        target_fail_rate = random.uniform(0.05, 0.25)
    else:
        target_fail_rate = random.uniform(0.25, 1.0)

    result = []

    for attack in selected_attacks:
        variations = attack["variations"]

        # Separate passed and failed variations
        passed_variations = [v for v in variations if v["status"] == "passed"]
        failed_variations = [v for v in variations if v["status"] == "failed"]

        # Decide total number of variations to generate for this attack
        variation_count = random.randint(10, 50)  # now can be large

        # Calculate how many should be failed to hit roughly the target_fail_rate
        failed_count = int(variation_count * target_fail_rate)
        passed_count = variation_count - failed_count

        # Sample with replacement to allow duplicates
        selected_variations = (
            random.choices(passed_variations, k=passed_count if passed_variations else 0) +
            random.choices(failed_variations, k=failed_count if failed_variations else 0)
        )

        random.shuffle(selected_variations)

        result.append({
            "attack_name": attack["attack_name"],
            "variations": selected_variations
        })

    return result

In [21]:
def generate_run():
    """
    Generate a run with the same structure, but only keep attack names and stats.
    Messages are removed for brevity.
    """
    run_id = str(uuid.uuid4())
    run_length_seconds = random.randint(40, 600)

    turn_type, injection_type, attacks = sample_attack_bucket()
    selected_attacks = sample_attacks(attacks)

    # Flatten all variations to calculate totals
    all_variations = [
        v
        for attack in selected_attacks
        for v in attack["variations"]
    ]

    total_runs = len(all_variations)
    total_passed = sum(1 for v in all_variations if v["status"] == "passed")
    total_failed = sum(1 for v in all_variations if v["status"] == "failed")
    total_error = 0  # placeholder for error state

    overall_fail_rate = total_failed / total_runs if total_runs else 0

    # Create simplified attacks list with only names and summary stats
    attacks_summary = []
    for attack in selected_attacks:
        attack_variations = attack["variations"]
        attack_total = len(attack_variations)
        attack_passed = sum(1 for v in attack_variations if v["status"] == "passed")
        attack_failed = sum(1 for v in attack_variations if v["status"] == "failed")

        attacks_summary.append({
            "attack_name": attack["attack_name"],
            "total_variations": attack_total,
            "passed": attack_passed,
            "failed": attack_failed,
            "overall_status": (
                "secure" if attack_failed / attack_total < 0.05 else
                "concerning" if attack_failed / attack_total < 0.25 else
                "critical"
            )
        })

    return {
        "evaluation_scope": {
            "turn_type": turn_type,
            "injection_type": injection_type
        },
        "aggregate_summary": {
            "total_runs": total_runs,
            "total_passed": total_passed,
            "total_failed": total_failed,
            "total_error": total_error,
            "overall_fail_rate": round(overall_fail_rate, 4),
            "overall_status": (
                "secure" if overall_fail_rate < 0.05 else
                "concerning" if overall_fail_rate < 0.25 else
                "critical"
            )
        },
        "attacks": attacks_summary
    }

In [23]:
# --- Output directory ---
base_data_dir = Path(os.pardir) / "data"
output_dir_generated = base_data_dir / "generated_runs"

def increase_dataset_size(directory=output_dir_generated, amount=20):
    Path(directory).mkdir(parents=True, exist_ok=True)
    for _ in range(amount):
        run = generate_run()
        file_id = str(uuid.uuid4())
        out_path = Path(directory) / f"{file_id}.generated.json"
        with open(out_path, "w", encoding="utf8") as f:
            json.dump(run, f, indent=2)
    print(f"Generated {amount} synthetic evaluation summaries in {directory}")

## Generate csv file from summarized and generated outputs

Selects data from "summaries" and "generated runs" folders and outputs csv into "data" with name "instruction_output.csv"

In [24]:
def generate_csv(input_data):
    """
    Generate a CSV with two columns:
    - instruction: the raw JSON or JSONL content
    - output: a human-readable summary of the report
    """
    folder_a, folder_b = input_data
    files_to_process = []

    for folder in [folder_a, folder_b]:
        if os.path.isdir(folder):
            for name in os.listdir(folder):
                path = os.path.join(folder, name)
                if name.lower().endswith((".json", ".jsonl")):
                    files_to_process.append(path)

    if not files_to_process:
        print("No JSON or JSONL files found.")
        return None

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["instruction", "output"])
        writer.writeheader()

        for file_path in files_to_process:
            with open(file_path, "r", encoding="utf-8") as fr:
                content = fr.read().strip()

            # Determine JSON or JSONL
            if file_path.lower().endswith(".json"):
                try:
                    reports = [json.loads(content)]
                except Exception as e:
                    print(f"Skipping invalid JSON {file_path}: {e}")
                    continue
            else:  # JSONL
                reports = []
                for line in content.splitlines():
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        reports.append(json.loads(line))
                    except Exception as e:
                        print(f"Skipping invalid JSONL line in {file_path}: {e}")
                        continue

            for report in reports:
                output_text = generate_report_string(report)
                writer.writerow({
                    "instruction": content,
                    "output": output_text
                })

    return csv_path


## Run code

In [ ]:
# Randomize all JSONL files in the raw folder
#for file in os.listdir(raw_dir):
#    file_path = os.path.join(raw_dir, file)
#    if os.path.isfile(file_path) and file.endswith(".jsonl"):
#        randomize_report(file_path)

In [ ]:
# Summarize all processed files
#for file in os.listdir(processed_dir):
#    file_path = os.path.join(processed_dir, file)
#    if os.path.isfile(file_path) and file.endswith(".jsonl"):
#        summarize_SET_report("file", file_path)

In [40]:
# Increase dataset size by amount entries
increase_dataset_size(amount=50)

Generated 50 synthetic evaluation summaries in ../data/generated_runs


In [ ]:
# Generate CSV from the two folders
# generate_csv((csv_path, output_dir_generated))